# RSA on GPU (steps 1, 2, 4)

Runs the crossnobis searchlight (step 1) and model comparisons (steps 2 & 4) for **one participant × many RSA models** on a Colab GPU, then writes one `result_*.zip` per finished part to a Google Drive folder.

**Runtime:** pick **GPU** (L4 or T4) with **High-RAM** (Runtime → Change runtime type).

**Steps:**
1. Upload the `pkg_*.zip` built by `tools/create_package.py` to a Drive folder.
2. Set `PKG_ZIP` (path to that zip) and `OUT_DIR` (a Drive output folder) below.
3. Run all cells. Result zips appear in `OUT_DIR` (`result_step1_*.zip` once, then `result_<model>_*.zip` per model). Re-running skips models already done.
4. Back on the workstation: `tools/unpack_results.py` merges the results onto the pipeline disk.

In [ ]:
# 1. Check the GPU and install nibabel (torch is preinstalled on Colab).
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!pip -q install nibabel

In [ ]:
# 2. Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. EDIT THESE: the package zip on Drive, and where results should go.
PKG_ZIP = '/content/drive/MyDrive/rsa_colab/pkg_H-sub-40_EmoC_basic-block.zip'
OUT_DIR = '/content/drive/MyDrive/rsa_colab/results'
BATCH   = 1024   # searchlight voxel batch; lower if you hit out-of-memory

# Force step 1 to recompute even if the package bundles maps (or its manifest
# says step1_done) -- e.g. maps built pre-alignment (before step 0.5).
CALCULATE_STEP1 = False
# Also delete any step-1 maps already unpacked into the package before deciding
# -- avoids stale maps sitting next to freshly computed ones under the other
# pair orientation. On its own this already forces a recompute.
DELETE_STEP1 = False

# Optional: overwrite the package's bundled code/run_colab.py (and gpu_rsa.py)
# with newer copies from Drive before importing, so fixes/new flags reach an
# already-built package without rebuilding and re-uploading its zip. Leave
# empty ('') to skip and just use whatever the package already bundles.
CODE_OVERRIDE = '/content/drive/MyDrive/rsa_colab/run_colab.py'
GPU_RSA_OVERRIDE = '/content/drive/MyDrive/rsa_colab/gpu_rsa.py'

In [ ]:
# 4. Unzip the package into Colab-local storage (fast local disk).
import os, zipfile, shutil
PKG_ROOT = '/content/pkg'
if os.path.isdir(PKG_ROOT):
    shutil.rmtree(PKG_ROOT)
os.makedirs(PKG_ROOT, exist_ok=True)
with zipfile.ZipFile(PKG_ZIP) as zf:
    zf.extractall(PKG_ROOT)
print('unpacked to', PKG_ROOT)
print(sorted(os.listdir(PKG_ROOT)))

In [ ]:
# 5. Optionally overwrite the package's bundled run_colab.py/gpu_rsa.py from
# Drive -- lets fixes/new flags reach a package built before this notebook cell
# existed, without rebuilding it. gpu_rsa.py matters too: run_colab.py imports
# it, and functions added there (e.g. step-1 zip naming) must come from the
# same override, or run_colab.py will call into the package's stale copy. No-op
# for any override left empty or missing on Drive.
for _override, _dest in ((CODE_OVERRIDE, 'run_colab.py'), (GPU_RSA_OVERRIDE, 'gpu_rsa.py')):
    if _override and os.path.exists(_override):
        shutil.copy(_override, os.path.join(PKG_ROOT, 'code', _dest))
        print(f'overrode code/{_dest} from', _override)

In [ ]:
# 6. Run step 1 (once) + steps 2/4 for every model. Resumable: skips finished models.
import sys
sys.path.insert(0, os.path.join(PKG_ROOT, 'code'))
import run_colab
written = run_colab.run_package(PKG_ROOT, OUT_DIR, batch=BATCH, verbose=True,
                                 calculate_step1=CALCULATE_STEP1,
                                 delete_step1=DELETE_STEP1)
print('\nnew result zips:')
for w in written:
    print(' ', w)